# ENAHO — Exploración de la estructura de microdatos

**Issue #13** · `area:datos` · Milestone M3

Objetivo: entender la estructura de los microdatos de la **ENAHO** (Encuesta Nacional de Hogares, INEI) para poder, en las issues siguientes, derivar la canasta básica de consumo por departamento.

- **#13 (esta)** — explorar estructura: módulos, columnas, códigos de producto, factor de expansión.
- **#17** — diseñar la tabla `canasta_consumo_dept`.
- **#19** — construir la canasta básica representativa por departamento.

Productos objetivo del MVP: **papa, limón, pollo, cebolla, huevo, tomate**.

---

## Cómo se obtienen los datos

La descarga se hace con la librería **`enahodata`** (celda de más abajo), que baja y descomprime los módulos directamente del INEI. **Funciona desde una IP peruana** — el geo-bloqueo del INEI solo afecta a los runners de GitHub, no a tu máquina.

En el sistema de microdatos del INEI los módulos se identifican **por número** (no por el "601"):

| Módulo | Nº en el sistema | Archivo descargado |
|---|---|---|
| Gastos en Alimentos y Bebidas | **07** | `...-601.dta` |
| Sumaria (variables calculadas por hogar) | **34** | `sumaria-<anio>.dta` |

`enahodata` baja en formato **`.dta` (Stata)**; `pyreadstat.read_dta` lo lee preservando las etiquetas de valor (las que traducen los códigos de producto).

Los datos viven en `data/enaho/` (carpeta **ignorada por git** — nunca se versiona). Opcionalmente se respaldan en R2 (`enaho/<ANIO>/`) para que #17/#19 puedan correr en CI, donde el INEI sí está geo-bloqueado.

## 0. Setup y parámetros

In [ ]:
from pathlib import Path

import pandas as pd
import pyreadstat
from dotenv import load_dotenv

load_dotenv()  # credenciales R2 desde el .env de la raíz del repo

# Año de la ENAHO a explorar.
# - 2023 funciona ya (es el último que conoce enahodata v0.0.3).
# - Para 2024/2025 hay que aportar el "código INEI" del año en la celda de
#   descarga (CODIGOS_INEI) — ver instrucciones ahí.
# Para la issue #13 (estructura) cualquier año reciente sirve: módulos, columnas
# y códigos de producto son idénticos. El año solo importa para los pesos de la
# canasta (#19).
ANIO = 2023

# Códigos de módulo en el sistema de microdatos del INEI (NO es el "601"):
#   07 = Gastos en Alimentos y Bebidas  → archivo ...-601.dta
#   34 = Sumaria (variables calculadas por hogar)
MOD_ALIMENTOS = "07"
MOD_SUMARIA = "34"

# Carpeta local (gitignored) donde enahodata descarga y descomprime los datos.
DIR_DATA = Path.cwd().parent / "data" / "enaho"
DIR_DATA.mkdir(parents=True, exist_ok=True)

# Prefijo de las claves en R2 (respaldo archival opcional, ver sección 4).
PREFIJO_R2 = f"enaho/{ANIO}/"


def col(df, nombre):
    """Resuelve un nombre de columna ignorando mayúsculas/minúsculas.

    Los .dta de Stata suelen traer los nombres en minúscula (p601a, ubigeo),
    mientras que en SPSS van en mayúscula. Esto unifica ambos casos.
    """
    indice = {c.lower(): c for c in df.columns}
    return indice.get(nombre.lower())


print(f"ANIO={ANIO}  ·  módulos {MOD_ALIMENTOS} (601) + {MOD_SUMARIA} (sumaria)")
print(f"Datos en: {DIR_DATA}")

### Descarga con `enahodata`

Baja y descomprime los módulos 07 (601) y 34 (sumaria) directamente del INEI. Solo descarga si los `.dta` no están ya en `data/enaho/`. Después localiza las rutas reales con `rglob` (la librería crea subcarpetas propias).

In [ ]:
import enahodata.enahodata as _ed
from enahodata import enahodata

# enahodata v0.0.3 solo conoce años hasta 2023 en su tabla interna (YEAR_MAP).
# Para 2024/2025 hay que aportar el "código INEI" del año y se inyecta aquí.
#
# Cómo obtener el código (desde una IP peruana):
#   1. Ir a https://proyectos.inei.gob.pe/microdatos/
#   2. Encuesta: ENAHO - Metodología ACTUALIZADA → elegir el año → Módulo 07.
#   3. Al iniciar la descarga, la URL es:
#        .../iinei/srienaho/descarga/STATA/<CODIGO>-Modulo07.zip
#      Copiar ese <CODIGO> (un número) aquí abajo.
#   (Para referencia, enahodata ya trae: 2021→759, 2022→784, 2023→906.)
CODIGOS_INEI = {
    # "2024": 0,   # <-- pegar el código real de 2024 y poner ANIO = 2024 arriba
    # "2025": 0,
}
for anio_str, codigo in CODIGOS_INEI.items():
    _ed.YEAR_MAP[anio_str] = {"codigo": codigo, "year": int(anio_str)}

if str(ANIO) not in _ed.YEAR_MAP:
    raise ValueError(
        f"El año {ANIO} no está en enahodata. Agregá su código INEI en "
        f"CODIGOS_INEI (ver instrucciones arriba) o usá un año soportado: "
        f"{sorted(_ed.YEAR_MAP)}."
    )


def localizar(patron: str) -> Path:
    """Devuelve el primer .dta que cumpla el patrón (búsqueda recursiva en DIR_DATA)."""
    encontrados = sorted(DIR_DATA.rglob(patron))
    if not encontrados:
        raise FileNotFoundError(
            f"No se encontró ningún archivo {patron!r} bajo {DIR_DATA}. "
            "¿Terminó la descarga? ¿Es correcto el año/código?"
        )
    return encontrados[0]


# Descarga sólo si todavía no hay un .dta del módulo 601 en local.
if not list(DIR_DATA.rglob("*601*.dta")):
    enahodata(
        modulos=[MOD_ALIMENTOS, MOD_SUMARIA],
        anios=[str(ANIO)],
        descomprimir=True,
        only_dta=True,
        overwrite=False,
        output_dir=str(DIR_DATA),
        panel=False,
    )

ruta_601 = localizar("*601*.dta")
ruta_sumaria = localizar("sumaria*.dta")
print("601    :", ruta_601)
print("sumaria:", ruta_sumaria)

## 1. Módulo 601 — Gastos en Alimentos y Bebidas

Es el módulo central para la canasta: una fila por (hogar × producto consumido), con el código del producto, el gasto y la cantidad. Cargamos **datos + metadatos** (las etiquetas son las que traducen los códigos numéricos).

In [ ]:
df601, meta601 = pyreadstat.read_dta(str(ruta_601))

print(f"Filas: {len(df601):,}  ·  Columnas: {df601.shape[1]}")
df601.head()

### 1.1 Diccionario de columnas

`meta.column_names_to_labels` es el diccionario oficial: nombre técnico → descripción legible. Lo dejamos a la vista para identificar las columnas que importan (código de producto, gasto, cantidad, geografía, factor de expansión).

In [ ]:
diccionario_601 = pd.DataFrame(
    meta601.column_names_to_labels.items(),
    columns=["columna", "etiqueta"],
)
pd.set_option("display.max_rows", None)
diccionario_601

In [ ]:
# Columnas candidatas que esperamos usar aguas abajo. Se resuelven con col()
# para ignorar mayúsc./minúsc. (los .dta de Stata suelen ir en minúscula).
# VERIFICAR los nombres exactos contra el diccionario de arriba:
#   - Identificadores de hogar : CONGLOME, VIVIENDA, HOGAR
#   - Geografía                : UBIGEO (6 díg.; 2 primeros = departamento), DOMINIO
#   - Producto                 : P601A  (código del producto/ítem de la canasta)
#   - Gasto / cantidad         : columnas I601* / D601* / G601* (gasto en soles)
#   - Factor de expansión      : FACTOR07 o FACTORA07 (factor anual)
candidatas = ["CONGLOME", "VIVIENDA", "HOGAR", "UBIGEO", "DOMINIO",
              "P601A", "FACTOR07", "FACTORA07"]
presentes = [col(df601, c) for c in candidatas if col(df601, c)]
ausentes = [c for c in candidatas if not col(df601, c)]
print("Presentes:", presentes)
print("Ausentes (buscar equivalente en el diccionario):", ausentes)

### 1.2 Etiquetas de valor (códigos de producto)

`meta.variable_value_labels` mapea cada código numérico a su nombre. La columna de producto (`P601A`) trae aquí el catálogo completo de alimentos: es donde localizaremos papa, limón, pollo, cebolla, huevo y tomate.

In [ ]:
# Resuelve el nombre real (p601a / P601A) sin importar la capitalización.
COL_PRODUCTO = col(df601, "P601A")

etiquetas_producto = meta601.variable_value_labels.get(COL_PRODUCTO, {})
print(f"{COL_PRODUCTO}: {len(etiquetas_producto)} códigos de producto")

catalogo = pd.DataFrame(
    etiquetas_producto.items(), columns=["codigo", "producto"]
).sort_values("codigo")
catalogo.head(30)

### 1.3 Mapeo de los 6 productos del MVP

Buscamos por nombre en el catálogo para fijar los códigos exactos que usaremos en #17/#19. Un mismo alimento puede tener varias presentaciones (p. ej. *papa blanca*, *papa amarilla*): hay que decidir cuáles agrupar.

In [ ]:
objetivos = ["papa", "limon", "limón", "pollo", "cebolla", "huevo", "tomate"]
patron = "|".join(objetivos)

coincidencias = catalogo[
    catalogo["producto"].str.normalize("NFKD")
    .str.encode("ascii", "ignore").str.decode("ascii")
    .str.contains(patron, case=False, na=False)
]
coincidencias

## 2. Sumaria — gasto agregado por hogar

Una fila por hogar, con totales de gasto/ingreso ya calculados por el INEI. Sirve para normalizar (peso del gasto en alimentos sobre el total) y validar agregaciones.

In [ ]:
df_sum, meta_sum = pyreadstat.read_dta(str(ruta_sumaria))
print(f"Filas (hogares): {len(df_sum):,}  ·  Columnas: {df_sum.shape[1]}")
pd.DataFrame(meta_sum.column_names_to_labels.items(), columns=["columna", "etiqueta"]).head(40)

## 3. Factor de expansión

**Crítico.** Cada hogar de la muestra representa a muchos hogares de la población. Para obtener cifras representativas hay que ponderar por el **factor de expansión** (`FACTOR07` / `FACTORA07`), nunca promediar en crudo. Demostramos la agregación correcta por departamento (2 primeros dígitos del `UBIGEO`).

In [ ]:
# Resuelve los nombres reales (mayúsc./minúsc.) confirmados en 1.1.
COL_FACTOR = col(df601, "FACTORA07") or col(df601, "FACTOR07")
COL_UBIGEO = col(df601, "UBIGEO")

df = df601.copy()
df["departamento"] = df[COL_UBIGEO].astype(str).str.zfill(6).str[:2]

# Nº de hogares (muestra) vs. población representada (ponderada) por departamento.
resumen_dept = (
    df.groupby("departamento")
    .agg(
        filas_muestra=(COL_FACTOR, "size"),
        poblacion_estimada=(COL_FACTOR, "sum"),
    )
    .sort_index()
)
resumen_dept

## 4. (Opcional) Respaldo de los crudos en R2

Como `enahodata` ya hace la descarga reproducible desde cualquier IP peruana, R2 es ahora un **respaldo opcional**. Vale la pena sobre todo si #17/#19 llegan a correr en **CI**: los runners de GitHub no pueden alcanzar al INEI (geo-bloqueo), así que bajarían los `.dta` de R2 en su lugar. La ENAHO de un año es inmutable → un único upload. Cambiar `SUBIR = True` para ejecutarlo.

In [ ]:
import os

SUBIR = False

if SUBIR:
    from observatorio.carga.r2_a_supabase import cliente_r2

    s3 = cliente_r2()
    bucket = os.environ["R2_BUCKET"]
    for dta in (ruta_601, ruta_sumaria):
        clave = PREFIJO_R2 + dta.name
        print(f"⬆️  Subiendo {dta.name} → {clave}")
        s3.upload_file(str(dta), bucket, clave)
    print("✅ Respaldo en R2 completo.")
else:
    print("SUBIR=False — respaldo opcional. Cambiar a True para subir los .dta a R2.")

## 5. Hallazgos → `docs/enaho.md`

Destilar aquí las conclusiones de la exploración y trasladarlas a [`docs/enaho.md`](../docs/enaho.md), que es lo que alimenta el diseño de la tabla (#17):

- Año usado y archivos (`.dta`) cargados.
- Nombres reales confirmados de: columna de producto, gasto, cantidad, geografía, factor de expansión.
- Códigos ENAHO definitivos de los 6 productos del MVP (y qué presentaciones se agrupan).
- Granularidad de la agregación por departamento y uso del factor de expansión.
- Dudas / decisiones pendientes para #17.